# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections in order — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a **ranking** task. My goal is to order content pages by how urgently they should be reviewed or refreshed. Ranking fits this lane because the action is not just yes/no; the real decision is which pages should be handled first when time is limited.

In [ ]:
import pandas as pd
from pathlib import Path

possible_paths = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('data/content_refresh_anonymized.csv'),
    Path('content_refresh_anonymized.csv')
]

csv_path = next((p for p in possible_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not find the starter CSV. Update the path in this cell to the correct dataset file.')

df = pd.read_csv(csv_path)
df.head()

## 2. Target or proxy

My target is a review-priority label or score for each page. If a direct label is not available, I can use a proxy based on observed signals such as declining trend, low CTR, weak engagement, or a combination of those indicators.

The point is to predict which pages should be reviewed first, not to predict traffic perfectly.

In [ ]:
target_cols = [c for c in ['is_declining', 'needs_ctr_fix', 'needs_engagement_fix', 'health_score', 'trend_pct', 'ctr', 'avg_position', 'engagement_rate'] if c in df.columns]
target_cols

## 3. Success metric

A good metric for this problem is **Precision@K** or another top-of-list ranking metric, because the team only has time to work on a small number of pages first. If the highest-priority pages at the top of the list are truly the ones that need work, the model is useful.

This metric matches the action: the team will not review every page, only the top-ranked ones.

In [ ]:
rows = len(df)
clients = df['client_id'].nunique() if 'client_id' in df.columns else None
cols = df.shape[1]

print(f'Rows: {rows}')
print(f'Clients: {clients}')
print(f'Columns: {cols}')

## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one page**. One row should represent one content item that can be scored or ranked for review.

The dataframe below shows that unit directly so I can see what one row contains before I define features or a label.

In [ ]:
df.head(1)

## 5. Why ML beats a fixed rule here

A fixed rule would be too blunt because page performance is influenced by several signals at once, and those signals can interact differently across pages. A rule like 'refresh every page with low CTR' would miss context, while a model can learn patterns from many examples and produce a better priority ranking.

This is an ML problem because I want a data-driven ranking that improves over a simple hand-written rule, not a one-size-fits-all cutoff.

In [ ]:
signals = {}
for col in ['is_declining', 'needs_ctr_fix', 'needs_engagement_fix']:
    if col in df.columns:
        signals[col] = float(df[col].mean())
for col in ['ctr', 'avg_position', 'engagement_rate', 'health_score', 'trend_pct']:
    if col in df.columns:
        signals[f'{col}_mean'] = float(df[col].mean())

signals

## Self-check

- [ ] I named the ML task type.
- [ ] I named the target or proxy.
- [ ] I named the success metric.
- [ ] I showed the unit of analysis as a real dataframe.
- [ ] I explained why ML is better than a fixed rule here.
- [ ] I tied the output to a real content action.